# Week 7 — Day 2: Prompt Data and Base Model

Prepare the dataset for fine-tuning:
1. Load the cleaned items dataset from Hugging Face Hub
2. Tokenize each item's summary with the Llama 3.2 tokenizer
3. Inspect the token-count distribution and pick a cutoff to truncate outliers
4. Build prompts (with completions for train/val, without for test)
5. Push the prompt-formatted dataset back to the Hub for use during training

## Imports & login

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

In [ ]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

## Load the dataset from Hugging Face Hub

Replace `username` with your own Hugging Face username — this is where your prepared `items_full` / `items_lite` dataset is hosted.

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

## Load the base model's tokenizer

Token counts are tokenizer-specific — same text, different tokenizer → different counts. We use the Llama 3.2 tokenizer because that's the model we'll fine-tune.

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

In [ ]:
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel('Number of tokens in summary')
plt.ylabel('Count')
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.show()

## Pick a token cutoff

Capping at 110 tokens truncates the long-tail outliers without losing too much of the dataset.

In [ ]:
CUTOFF = 110
cut = len([count for count in token_counts if count > CUTOFF])
print(f"With this CUTOFF, we will truncate {cut:,} items which is {cut/len(items):.1%}")

In [ ]:
print(train[0].summary)

## Build prompts for each item

- Train and validation items get `prompt + completion` (the model sees the answer during training).
- Test items get only the prompt (we'll evaluate generations against held-out completions).

In [ ]:
for item in tqdm(train + val):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)

In [ ]:
print("PROMPT:")
print(test[0].prompt)
print("COMPLETION:")
print(test[0].completion)

## Sanity-check the full-prompt token distribution

In [ ]:
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel('Number of tokens in prompt and the completion')
plt.ylabel('Count')
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.show()

## Push the prompt-formatted dataset back to Hugging Face Hub

These uploaded datasets get loaded by the Colab training notebook on day 3.

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
dataset = f"{username}/items_prompts_lite" if LITE_MODE else f"{username}/items_prompts_full"

Item.push_prompts_to_hub(dataset, train, val, test)